# 자기장 데이터셋 EDA (Day 7)

`data/processed/zones_dataset.csv`를 불러와 맵별 분포와 자기장 축소 패턴을 살펴본다.

목적: 베이스라인 모델(Week 2)을 만들기 전에 데이터의 생김새를 파악한다.
- 맵별로 phase가 몇 개씩 있는지
- 단계가 올라갈수록 반경이 어떻게 줄어드는지 (축소 비율)
- 현재 원(safety) 중심이 다음 원(poison) 중심으로 얼마나 이동하는지

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 노트북이 ml/ 폴더에 있으므로 상위의 data/를 참조
CSV_PATH = os.path.join("..", "data", "processed", "zones_dataset.csv")
df = pd.read_csv(CSV_PATH)
print(f"행 수: {len(df)}, 고유 매치 수: {df['match_id'].nunique()}")
df.head()

## 1. 맵별 phase 분포

어떤 맵의 데이터가 많은지 확인한다. 회귀 모델은 맵마다 패턴이 다르므로,
데이터가 충분한 맵부터 학습 대상으로 삼는다.

In [ ]:
map_counts = df["map"].value_counts()
print(map_counts)

plt.figure(figsize=(8, 4))
sns.barplot(x=map_counts.index, y=map_counts.values)
plt.title("맵별 phase 개수")
plt.ylabel("phase count")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. 단계(phase)별 반경 축소 패턴

phase가 올라갈수록 다음 자기장 반경(poison_radius)이 줄어드는 모습을 본다.
베이스라인 모델의 핵심 가정: "단계별 축소 비율이 어느 정도 일정하다".

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="phase", y="poison_radius")
plt.title("phase별 다음 자기장 반경 분포")
plt.tight_layout()
plt.show()

## 3. 축소 비율 (다음 반경 / 현재 반경)

현재 안전지대 반경 대비 다음 자기장 반경의 비율. 1보다 작으면 줄어든다는 뜻.
베이스라인 모델이 이 비율의 평균을 그대로 적용하므로, 분포가 좁을수록 예측이 쉬워진다.

In [ ]:
# 0으로 나누는 것을 방지하며 축소 비율 계산
df_valid = df[df["safety_radius"] > 0].copy()
df_valid["shrink_ratio"] = df_valid["poison_radius"] / df_valid["safety_radius"]

print(df_valid["shrink_ratio"].describe())

plt.figure(figsize=(8, 4))
sns.histplot(df_valid["shrink_ratio"], bins=30, kde=True)
plt.title("축소 비율 분포 (poison_radius / safety_radius)")
plt.tight_layout()
plt.show()

## 4. 중심 이동 거리 (현재 원 → 다음 원)

다음 자기장 중심이 현재 안전지대 중심에서 얼마나 떨어져 있는지(유클리드 거리).
예측의 난이도를 가늠하는 지표 — 이동 거리가 클수록 예측이 어렵다.

In [ ]:
df["move_dist"] = np.sqrt(
    (df["poison_x"] - df["safety_x"]) ** 2 + (df["poison_y"] - df["safety_y"]) ** 2
)

print(df.groupby("phase")["move_dist"].mean())

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="phase", y="move_dist")
plt.title("phase별 중심 이동 거리")
plt.tight_layout()
plt.show()

## 요약

- 맵별 데이터 양이 불균형하므로, Week 3 회귀 모델은 데이터가 많은 맵부터 다룬다.
- 단계가 올라갈수록 반경이 줄고 이동 거리도 대체로 감소 → 베이스라인 가정과 부합.
- 다음 단계(Week 2): 이 통계를 이용해 규칙 기반 베이스라인 모델을 구현한다.